MLP 개발 흐름

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# 1. 데이터 준비: X=(8,2), y=(8,1)
X = torch.tensor([
    [0.1, 0.2],
    [0.2, 0.1],
    [0.3, 0.5],
    [0.4, 0.2],
    [0.6, 0.5],
    [0.7, 0.8],
    [0.8, 0.6],
    [0.9, 0.9]
], dtype=torch.float32)

y = torch.tensor([
    [0], [0], [0], [0],
    [1], [1], [1], [1]
], dtype=torch.float32)

# 2. 모델 정의: input layer: 2 → hidden layer 1: 4 → output layer: 1
model = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)

# 3. Loss / Optimizer 정의 
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

# 4. 학습 
for epoch in range(501):  # Epoch 0~500, 총 501번 업데이트
    optimizer.zero_grad()    # 이전 기울기 초기화
    pred = model(X)          # 4.1 순전파
    loss = criterion(pred, y)  # 4.2 손실 계산
    loss.backward()          # 4.3 역전파
    optimizer.step()         # 4.4 가중치 갱신

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# 5 테스트 (예측 결과 생성)
with torch.no_grad():
    probabilities = model(X)
    predictions = (probabilities >= 0.5).int()

print("예측:", predictions.squeeze().tolist())
print("정답:", y.int().squeeze().tolist())

Epoch 0, Loss: 0.7556
Epoch 100, Loss: 0.0122
Epoch 200, Loss: 0.0031
Epoch 300, Loss: 0.0014
Epoch 400, Loss: 0.0008
Epoch 500, Loss: 0.0005
예측: [0, 0, 0, 0, 1, 1, 1, 1]
정답: [0, 0, 0, 0, 1, 1, 1, 1]


DNN 역전파 구현

In [ ]:
import torch
import torch.nn as nn

# 재현 가능한 결과를 위한 난수 시드
torch.manual_seed(2)

# ----------------------
# 1. 입력 데이터와 실제 정답
# ----------------------

X = torch.tensor([
    [0.5, 0.3, 0.8],
    [0.2, 0.0, 0.3],
    [0.8, 0.4, 1.0],
    [0.3, 0.1, 0.5]
], dtype=torch.float32)

y = torch.tensor([
    [1.0],
    [0.0],
    [1.0],
    [0.0]
])

# ----------------------
# 2. DNN 모델 정의
# ----------------------

model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),

    nn.Linear(4, 2),
    nn.ReLU(),

    nn.Linear(2, 1)
)

# 이진 분류 손실 함수
loss_fn = nn.BCEWithLogitsLoss()

# 학습 모드
model.train()

# ----------------------
# 3. 순전파
# ----------------------

# 기존 Gradient 초기화
model.zero_grad()

# 현재 가중치 저장
weight_before = model[0].weight.detach().clone()

# Logit 계산
logits = model(X)

# 손실 계산
loss = loss_fn(logits, y)

print("입력 Shape:", tuple(X.shape))
print("Logit Shape:", tuple(logits.shape))
print("손실:", round(loss.item(), 4))

# ----------------------
# 4. 역전파
# ----------------------

loss.backward()

# ----------------------
# 5. Gradient 확인
# ----------------------

for name, parameter in model.named_parameters():
    print(
        f"{name} | "
        f"파라미터 Shape: {tuple(parameter.shape)} | "
        f"Gradient Shape: {tuple(parameter.grad.shape)} | "
        f"Gradient 크기: {parameter.grad.norm().item():.5f}"
    )

# ----------------------
# 6. 파라미터 변경 여부 확인
# ----------------------

weight_after = model[0].weight.detach()

is_changed = not torch.equal(
    weight_before,
    weight_after
)

print("역전파 후 가중치 변경 여부:", is_changed)

# 실행 결과 예시
# 입력 Shape: (4, 3)
# Logit Shape: (4, 1)
# 손실: 0.7115
# 0.weight | 파라미터 Shape: (4, 3) | Gradient Shape: (4, 3) | Gradient 크기: 0.06920
# 0.bias | 파라미터 Shape: (4,) | Gradient Shape: (4,) | Gradient 크기: 0.05926
# 2.weight | 파라미터 Shape: (2, 4) | Gradient Shape: (2, 4) | Gradient 크기: 0.04049
# 2.bias | 파라미터 Shape: (2,) | Gradient Shape: (2,) | Gradient 크기: 0.04378
# 4.weight | 파라미터 Shape: (1, 2) | Gradient Shape: (1, 2) | Gradient 크기: 0.00328
# 4.bias | 파라미터 Shape: (1,) | Gradient Shape: (1,) | Gradient 크기: 0.11058
# 역전파 후 가중치 변경 여부: False

pytorch로 기울기 소실과 폭주 확인

In [2]:
import torch

# ----------------------
# 기울기 전달 확인 함수
# ----------------------

def inspect_gradient(scale, depth=8):
    # Gradient를 추적할 입력값
    x = torch.tensor(1.0, requires_grad=True)

    # 단계별 출력값 기록
    values = [x]

    h = x

    # 동일한 연산을 여러 번 수행
    for _ in range(depth):
        h = h * scale

        # 중간 출력의 Gradient도 확인
        h.retain_grad()
        values.append(h)

    # 목표값과 손실 계산
    target = torch.tensor(1.0)
    loss = (h - target) ** 2

    # 역전파
    loss.backward()

    # 결과 확인
    print(f"\n곱하는 값: {scale}")
    print(f"연산 단계: {depth}")
    print(f"최종 출력: {h.item():.6f}")
    print(f"손실: {loss.item():.6f}")

    print(
        "마지막 출력 Gradient:",
        f"{values[-1].grad.item():.6f}"
    )

    print(
        "첫 번째 단계 Gradient:",
        f"{values[1].grad.item():.6f}"
    )

    print(
        "입력 Gradient:",
        f"{x.grad.item():.6f}"
    )


# ----------------------
# 1. 작은 값이 반복되는 경우
# ----------------------

inspect_gradient(scale=0.5)

# ----------------------
# 2. 큰 값이 반복되는 경우
# ----------------------

inspect_gradient(scale=2.0)


# 곱하는 값: 0.5
# 연산 단계: 8
# 최종 출력: 0.003906
# 손실: 0.992203
# 마지막 출력 Gradient: -1.992188
# 첫 번째 단계 Gradient: -0.015564
# 입력 Gradient: -0.007782

# 곱하는 값: 2.0
# 연산 단계: 8
# 최종 출력: 256.000000
# 손실: 65025.000000
# 마지막 출력 Gradient: 510.000000
# 첫 번째 단계 Gradient: 65280.000000
# 입력 Gradient: 130560.000000


곱하는 값: 0.5
연산 단계: 8
최종 출력: 0.003906
손실: 0.992203
마지막 출력 Gradient: -1.992188
첫 번째 단계 Gradient: -0.015564
입력 Gradient: -0.007782

곱하는 값: 2.0
연산 단계: 8
최종 출력: 256.000000
손실: 65025.000000
마지막 출력 Gradient: 510.000000
첫 번째 단계 Gradient: 65280.000000
입력 Gradient: 130560.000000


gradient 확인 및 clipping 구현

In [ ]:
import torch
import torch.nn as nn

# 재현 가능한 결과
torch.manual_seed(42)

# ----------------------
# 1. 입력 데이터
# ----------------------

X = torch.tensor([
    [0.5, 0.3, 0.8],
    [0.2, 0.0, 0.3],
    [0.8, 0.4, 1.0],
    [0.3, 0.1, 0.5]
], dtype=torch.float32)

y = torch.tensor([
    [1.0],
    [0.0],
    [1.0],
    [0.0]
])

# ----------------------
# 2. DNN 모델 정의
# ----------------------

model = nn.Sequential(
    nn.Linear(3, 6),
    nn.ReLU(),

    nn.Linear(6, 4),
    nn.ReLU(),

    nn.Linear(4, 1)
)

# 손실 함수
loss_fn = nn.BCEWithLogitsLoss()

# ----------------------
# 3. 순전파와 손실 계산
# ----------------------

model.zero_grad()

# 가중치 변경 여부 확인용
weight_before = model[0].weight.detach().clone()

logits = model(X)
loss = loss_fn(logits, y)

print("손실:", round(loss.item(), 4))

# ----------------------
# 4. 역전파
# ----------------------

loss.backward()

# ----------------------
# 5. 레이어별 Gradient 확인
# ----------------------

for name, parameter in model.named_parameters():
    if parameter.grad is not None:
        grad_norm = parameter.grad.norm().item()

        print(
            f"{name} | "
            f"Gradient Norm: {grad_norm:.5f}"
        )

# ----------------------
# 6. Gradient Clipping
# ----------------------

# Clipping 전 전체 Gradient Norm
grads = [
    p.grad
    for p in model.parameters()
    if p.grad is not None
]

total_norm_before = torch.linalg.vector_norm(
    torch.stack([grad.norm() for grad in grads])
)

# 설명을 위한 작은 기준값
max_norm = 0.03

# Gradient Clipping 적용
original_norm = nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=max_norm
)

# Clipping 후 전체 Gradient Norm
total_norm_after = torch.linalg.vector_norm(
    torch.stack([grad.norm() for grad in grads])
)

print(
    "Clipping 전:",
    round(total_norm_before.item(), 5)
)

print(
    "Clipping 후:",
    round(total_norm_after.item(), 5)
)

# ----------------------
# 7. 가중치 변경 여부 확인
# ----------------------

weight_changed = not torch.equal(
    weight_before,
    model[0].weight.detach()
)

print("가중치 변경 여부:", weight_changed)

# 실행 결과 예시
# 손실: 0.6871
# 0.weight | Gradient Norm: 0.02170
# 0.bias | Gradient Norm: 0.01429
# 2.weight | Gradient Norm: 0.02602
# 2.bias | Gradient Norm: 0.01865
# 4.weight | Gradient Norm: 0.04375
# 4.bias | Gradient Norm: 0.04481
# Clipping 전: 0.07498
# Clipping 후: 0.03
# 가중치 변경 여부: False

손실: 0.6871
0.weight | Gradient Norm: 0.02170
0.bias | Gradient Norm: 0.01429
2.weight | Gradient Norm: 0.02602
2.bias | Gradient Norm: 0.01865
4.weight | Gradient Norm: 0.04375
4.bias | Gradient Norm: 0.04481
Clipping 전: 0.07498
Clipping 후: 0.03
가중치 변경 여부: False


In [4]:
import torch

# 학습률에 따른 결과 비교
learning_rates = [0.05, 0.2, 0.8, 1.1]

for lr in learning_rates:
    # 동일한 초기 가중치
    w = torch.nn.Parameter(
        torch.tensor(1.0)
    )

    # SGD Optimizer
    optimizer = torch.optim.SGD(
        [w],
        lr=lr
    )

    losses = []

    # 5번 반복 업데이트
    for step in range(5):
        optimizer.zero_grad()

        # 손실 계산
        loss = (w - 3) ** 2

        # 역전파
        loss.backward()

        # 파라미터 업데이트
        optimizer.step()

        # 업데이트 후 손실 기록
        with torch.no_grad():
            current_loss = (w - 3) ** 2
            losses.append(current_loss.item())

    print(
        f"학습률={lr} | "
        f"최종 가중치={w.item():.4f} | "
        f"최종 손실={losses[-1]:.4f}"
    )

학습률=0.05 | 최종 가중치=1.8190 | 최종 손실=1.3947
학습률=0.2 | 최종 가중치=2.8445 | 최종 손실=0.0242
학습률=0.8 | 최종 가중치=3.1555 | 최종 손실=0.0242
학습률=1.1 | 최종 가중치=7.9766 | 최종 손실=24.7670
